In [ ]:
Lexical Richness — Twain and Reference Corpora

Purpose: Measures lexical diversity across the three Twain novels and both reference corpora (non-Twain child fiction; adult prose). Three complementary metrics are computed for each text and for each corpus as a whole:

TTR (Type-Token Ratio): types ÷ tokens. Simple but length-sensitive — longer texts always produce lower TTR, so it should not be used to compare texts of very different lengths.
MTLD (Measure of Textual Lexical Diversity): measures the mean length of sequential word strings that maintain a TTR above a threshold (0.72). Substantially more length-robust than TTR. Preferred metric for cross-text comparison.
HD-D (Hypergeometric Distribution D): probability-based measure estimating lexical diversity from random 42-word samples. Also length-robust; serves as a cross-check on MTLD.
Library: lexicalrichness — wraps McCarthy & Jarvis (2010) implementations.

Inputs: Three Twain novels + 8 child + 28 adult reference texts (plain-text UTF-8, Gutenberg editions).

Output: Per-text and per-corpus DataFrames; CSV files saved to BASE_PATH.

Used in: Chapter 3, lexical diversity comparison.

source: https://pypi.org/project/lexicalrichness/

In [3]:
%pip install lexicalrichness

  Using cached lexicalrichness-0.5.1-py3-none-any.whl
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 18.1 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.0/625.0 kB 12.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 15.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [lexicalrichness] [textblob]
Note: you may need to restart the kernel to use updated packages.


In [15]:
from pathlib import Path
import re
from lexicalrichness import LexicalRichness

BASE_PATH = Path(
    "/Users/manalaltammami/Desktop/twain_child_corpus"
)

source_files = [
    "Tom_Sawyer_clean.txt",
    "Huckleberry_Finn_clean.txt",
    "Prince_and_the_Pauper_clean.txt"
]

for filename in source_files:
    path = BASE_PATH / filename

    if not path.is_file():
        raise FileNotFoundError(f"File not found: {path}")

    print("Found:", filename)

Found: Tom_Sawyer_clean.txt
Found: Huckleberry_Finn_clean.txt
Found: Prince_and_the_Pauper_clean.txt


In [17]:
# Exact story boundaries checked against your supplied files.
rules = [
    (
        "Tom_Sawyer_clean.txt",
        "Tom_Sawyer_narrative.txt",
        r"(?m)^CHAPTER I[ \t]*$",
        None,
        r"(?m)^CHAPTER [IVXLCDM]+\.?[ \t]*\n",
        35
    ),
    (
        "Huckleberry_Finn_clean.txt",
        "Huckleberry_Finn_narrative.txt",
        r"(?m)^CHAPTER I\.[ \t]*\n(?:[ \t]*\n)+(?=You don.t know)",
        r"(?m)^THE END\. YOURS TRULY, _HUCK FINN_\.",
        r"(?m)^CHAPTER [IVXLCDM]+\.?[ \t]*\n|^CHAPTER THE LAST\.?[ \t]*\n",
        43
    ),
    (
        "Prince_and_the_Pauper_clean.txt",
        "Prince_and_the_Pauper_narrative.txt",
        r"(?m)^CHAPTER I\. The birth of the Prince and the Pauper\.[ \t]*$",
        r"(?m)^FOOTNOTES AND TWAIN’S NOTES[ \t]*$",
        r"(?m)^CHAPTER [IVXLCDM]+\. [^\n]*\n",
        33
    )
]

narrative_texts = {}

for source, output, start_rule, end_rule, heading_rule, expected in rules:
    with (BASE_PATH / source).open(encoding="utf-8-sig") as f:
        text = f.read()

    text = text.replace("\r\n", "\n").replace("\r", "\n")

    starts = list(re.finditer(start_rule, text))
    assert len(starts) == 1, f"Check opening boundary: {source}"

    start = starts[0].start()
    end = len(text)

    if end_rule:
        endings = list(re.finditer(end_rule, text[start:]))
        assert len(endings) == 1, f"Check closing boundary: {source}"
        end = start + endings[0].start()

    narrative = text[start:end]

    headings = list(re.finditer(heading_rule, narrative))
    assert len(headings) == expected, f"Check chapter count: {source}"

    narrative = re.sub(heading_rule, "", narrative)

    # Remove conclusion headings, but retain the conclusion itself.
    narrative = re.sub(
        r"(?m)^(?:CONCLUSION|Conclusion\. Justice and retribution\.)[ \t]*\n",
        "",
        narrative
    )

    # Remove numeric note references.
    narrative = re.sub(r"\{\d+\}", "", narrative)
    narrative = narrative.strip() + "\n"

    narrative_texts[output] = narrative
    (BASE_PATH / output).write_text(narrative, encoding="utf-8")

    print(f"\nCreated: {output}")
    print("Opening:", narrative[:120].replace("\n", " "))
    print("Ending:", narrative[-160:].strip().replace("\n", " "))
    print("Chapter headings removed:", len(headings))


Created: Tom_Sawyer_narrative.txt
Opening: “Tom!”  No answer.  “TOM!”  No answer.  “What’s gone with that boy, I wonder? You TOM!”  No answer.  The old lady pulled
Ending: nger ones again and see what sort of men and women they turned out to be; therefore it will be wisest not to reveal any of that part of their lives at present.
Chapter headings removed: 35

Created: Huckleberry_Finn_narrative.txt
Opening: You don’t know about me without you have read a book by the name of The Adventures of Tom Sawyer; but that ain’t no matt
Ending: I got to light out for the Territory ahead of the rest, because Aunt Sally she’s going to adopt me and sivilize me, and I can’t stand it. I been there before.
Chapter headings removed: 43

Created: Prince_and_the_Pauper_narrative.txt
Opening: In the ancient city of London, on a certain autumn day in the second quarter of the sixteenth century, a boy was born to
Ending: n of Edward VI. was a singularly merciful one for those harsh times.  Now that we are t

In [19]:
# Shared word-token rule:
# retain a, I, contractions, and internally hyphenated words.
TOKEN_PATTERN = re.compile(r"[a-z]+(?:['-][a-z]+)*")


def tokenize(text):
    return TOKEN_PATTERN.findall(
        text.lower().replace("’", "'")
    )


def lexical_metrics(tokens):
    lex = LexicalRichness(
        tokens,
        preprocessor=None,
        tokenizer=None
    )

    assert lex.wordlist == tokens
    assert lex.words == len(tokens)

    return {
        "Tokens": lex.words,
        "Types": lex.terms,
        "TTR": round(float(lex.ttr), 4),
        "MTLD": round(float(lex.mtld(threshold=0.72)), 2),
        "HD-D": round(float(lex.hdd(draws=42)), 4)
    }


twain_tokens = {}
twain_results = {}
combined_twain_tokens = []

for filename, text in narrative_texts.items():
    tokens = tokenize(text)

    twain_tokens[filename] = tokens
    twain_results[filename] = lexical_metrics(tokens)
    combined_twain_tokens.extend(tokens)

    print(filename)
    print(twain_results[filename])
    print()

twain_combined_results = lexical_metrics(combined_twain_tokens)

print("TWAIN COMBINED — NARRATIVE ONLY")
print(twain_combined_results)


Tom_Sawyer_narrative.txt
{'Tokens': 70518, 'Types': 7505, 'TTR': 0.1064, 'MTLD': 84.12, 'HD-D': 0.8688}

Huckleberry_Finn_narrative.txt
{'Tokens': 110729, 'Types': 6569, 'TTR': 0.0593, 'MTLD': 67.43, 'HD-D': 0.8511}

Prince_and_the_Pauper_narrative.txt
{'Tokens': 67007, 'Types': 7698, 'TTR': 0.1149, 'MTLD': 92.69, 'HD-D': 0.8615}

TWAIN COMBINED — NARRATIVE ONLY
{'Tokens': 248254, 'Types': 14419, 'TTR': 0.0581, 'MTLD': 77.52, 'HD-D': 0.8642}


In [25]:
from pathlib import Path
import re

BASE_PATH = Path(
    "/Users/manalaltammami/Desktop/twain_child_corpus"
)

# Chapter number followed by a separate chapter-title line.
two_line_heading = (
    r"(?m)^CHAPTER [A-Z-]+\.?[ \t]*\n"
    r"(?:[ \t]*\n)*[^\n]+\n"
)

child_rules = [
    (
        "ragged_dick.txt",
        "Ragged_Dick_narrative.txt",
        r"(?m)^CHAPTER I\.[ \t]*$",
        r"(?m)^\*\*\* END OF",
        two_line_heading,
        27
    ),
    (
        "Little_Women.txt",
        "Little_Women_narrative.txt",
        r"(?m)^CHAPTER ONE[ \t]*$",
        None,
        two_line_heading,
        47
    ),
    (
        "An_Old_Fashioned_Girl_verified_raw.txt",
        "An_Old_Fashioned_Girl_narrative.txt",
        r"(?m)^CHAPTER I\. POLLY ARRIVES[ \t]*$",
        r"(?m)^End of Project Gutenberg.s An Old-fashioned Girl",
        r"(?m)^CHAPTER [IVXLCDM]+\. [^\n]*\n",
        19
    ),
    (
        "toby_tylerclean.txt",
        "Toby_Tyler_narrative.txt",
        r"(?m)^CHAPTER I\.[ \t]*$",
        r"(?m)^\*\*\* END OF",
        two_line_heading,
        20
    ),
    (
        "five_little_peppers.txt",
        "Five_Little_Peppers_narrative.txt",
        r"(?m)^A HOME VIEW[ \t]*$",
        None,
        None,
        25
    ),
    (
        "hoosier_schoolboy.txt",
        "Hoosier_Schoolboy_narrative.txt",
        r"(?m)^CHAPTER I[ \t]*$",
        r"(?m)^End of Project Gutenberg.s The Hoosier School-boy",
        two_line_heading,
        27
    ),
    (
        "Little_Lord_Fauntleroy.txt",
        "Little_Lord_Fauntleroy_narrative.txt",
        r"(?m)^I\n\n(?=Cedric himself)",
        None,
        r"(?m)^[IVXLCDM]+[ \t]*\n(?=\n)",
        15
    ),
    (
        "The_Story_of_Patsy_verified_raw.txt",
        "The_Story_of_Patsy_narrative.txt",
        r"(?m)^CHAPTER I\.[ \t]*$",
        r"(?m)^\*\*\* END OF",
        two_line_heading,
        7
    )
]

pepper_titles = (
    "A HOME VIEW|MAKING HAPPINESS FOR MAMSIE|MAMSIE'S BIRTHDAY|"
    "TROUBLE FOR THE LITTLE BROWN HOUSE|MORE TROUBLE|"
    "HARD DAYS FOR POLLY|THE CLOUD OVER THE LITTLE BROWN HOUSE|"
    "JOEL'S TURN|SUNSHINE AGAIN|A THREATENED BLOW|SAFE|"
    "NEW FRIENDS|PHRONSIE PAYS A DEBT OF GRATITUDE|"
    "A LETTER TO JASPER|JOLLY DAYS|"
    "GETTING A CHRISTMAS FOR THE LITTLE ONES|CHRISTMAS BELLS!|"
    "EDUCATION AHEAD|BRAVE WORK AND THE REWARD|"
    "POLLY IS COMFORTED|PHRONSIE|"
    "GETTING READY FOR MAMSIE AND THE BOYS|"
    "WHICH TREATS OF A GOOD MANY MATTERS|"
    "POLLY'S DISMAL MORNING|POLLY'S BIG BUNDLE"
).split("|")

pepper_heading = (
    r"(?m)^(?:"
    + "|".join(re.escape(title) for title in pepper_titles)
    + r")[ \t]*\n"
)

child_narrative_texts = {}

for source, output, start_rule, end_rule, heading_rule, expected in child_rules:

    # Match the filename regardless of capitalization.
    matches = [
        path for path in BASE_PATH.iterdir()
        if path.is_file()
        and path.name.casefold() == source.casefold()
    ]
    assert len(matches) == 1, f"Check source filename: {source}"

    with matches[0].open(encoding="utf-8-sig") as f:
        text = f.read()

    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Locate the actual story opening.
    starts = list(re.finditer(start_rule, text))
    assert len(starts) == 1, f"Check opening: {source}"

    start = starts[0].start()
    end = len(text)

    # Locate the closing boundary where one is present.
    if end_rule:
        endings = list(re.finditer(end_rule, text[start:]))
        assert len(endings) == 1, f"Check ending: {source}"
        end = start + endings[0].start()

    narrative = text[start:end]

    if source == "five_little_peppers.txt":
        heading_rule = pepper_heading

    # Verify the expected number of chapters.
    headings = list(re.finditer(heading_rule, narrative))
    assert len(headings) == expected, (
        f"{source}: expected {expected} headings, "
        f"found {len(headings)}"
    )

    if source == "The_Story_of_Patsy_verified_raw.txt":
        # Remove chapter headings and their opening epigraphs.
        epigraph_rule = (
            heading_rule
            + r"(?:[ \t]*\n)*"
            + r"[ \t]+[^\n]*(?:\n[ \t]+[^\n]*)*\n"
        )

        epigraphs = list(re.finditer(epigraph_rule, narrative))
        assert len(epigraphs) == 7, "Check Patsy's epigraphs."

        narrative = re.sub(epigraph_rule, "\n", narrative)
    else:
        narrative = re.sub(heading_rule, "\n", narrative)

    # Retain both parts of Little Women; remove the part heading.
    if source == "Little_Women.txt":
        narrative = re.sub(
            r"(?m)^PART [12][ \t]*$",
            "",
            narrative
        )

    # Remove illustration labels and captions.
    narrative = re.sub(
        r"\[Illustration[^\]]*\]",
        "",
        narrative,
        flags=re.IGNORECASE
    )

    narrative = narrative.strip() + "\n"

    # Stop if a Gutenberg footer remains.
    assert not re.search(
        r"(?im)^(?:\*\*\*\s*END OF|End of Project Gutenberg)",
        narrative
    ), f"Gutenberg footer remains: {source}"

    child_narrative_texts[output] = narrative

    # Save a separate narrative copy; leave the source unchanged.
    (BASE_PATH / output).write_text(
        narrative,
        encoding="utf-8"
    )

    print(f"\nCreated: {output}")
    print("Opening:", narrative[:120].replace("\n", " "))
    print(
        "Ending:",
        narrative[-160:].strip().replace("\n", " ")
    )
    print("Chapter headings removed:", len(headings))

print("\nFinished: all eight narrative files created.")


Created: Ragged_Dick_narrative.txt
Opening: “Wake up there, youngster,” said a rough voice.  Ragged Dick opened his eyes slowly, and stared stupidly in the face of 
Ending: y life will find his history continued in a new volume, forming the second of the series, to be called,—  FAME AND FORTUNE; OR, THE PROGRESS OF RICHARD HUNTER.
Chapter headings removed: 27

Created: Little_Women_narrative.txt
Opening: “Christmas won’t be Christmas without any presents,” grumbled Jo, lying on the rug.  “It’s so dreadful to be poor!” sigh
Ending: ace and voice full of motherly love, gratitude, and humility...  “Oh, my girls, however long you may live, I never can wish you a greater happiness than this!”
Chapter headings removed: 47

Created: An_Old_Fashioned_Girl_narrative.txt
Opening: “IT'S time to go to the station, Tom.”  “Come on, then.”  “Oh, I'm not going; it's too wet. Should n't have a crimp left
Ending: pen dares not proceed, but pauses here, concluding in the words of the dear old fairy tal

In [27]:
child_tokens = {}
child_results = {}
combined_child_tokens = []

for filename, text in child_narrative_texts.items():
    tokens = tokenize(text)

    child_tokens[filename] = tokens
    child_results[filename] = lexical_metrics(tokens)
    combined_child_tokens.extend(tokens)

    print(filename)
    print(child_results[filename])
    print()

child_combined_results = lexical_metrics(combined_child_tokens)

print("NON-TWAIN CHILD FICTION — NARRATIVE ONLY")
print(child_combined_results)

Ragged_Dick_narrative.txt
{'Tokens': 47722, 'Types': 4692, 'TTR': 0.0983, 'MTLD': 100.1, 'HD-D': 0.8712}

Little_Women_narrative.txt
{'Tokens': 185825, 'Types': 11164, 'TTR': 0.0601, 'MTLD': 124.6, 'HD-D': 0.8787}

An_Old_Fashioned_Girl_narrative.txt
{'Tokens': 101818, 'Types': 7701, 'TTR': 0.0756, 'MTLD': 123.0, 'HD-D': 0.8783}

Toby_Tyler_narrative.txt
{'Tokens': 50233, 'Types': 4211, 'TTR': 0.0838, 'MTLD': 90.62, 'HD-D': 0.8538}

Five_Little_Peppers_narrative.txt
{'Tokens': 69400, 'Types': 4988, 'TTR': 0.0719, 'MTLD': 106.68, 'HD-D': 0.8795}

Hoosier_Schoolboy_narrative.txt
{'Tokens': 36080, 'Types': 4227, 'TTR': 0.1172, 'MTLD': 89.08, 'HD-D': 0.8639}

Little_Lord_Fauntleroy_narrative.txt
{'Tokens': 58627, 'Types': 4684, 'TTR': 0.0799, 'MTLD': 76.89, 'HD-D': 0.8562}

The_Story_of_Patsy_narrative.txt
{'Tokens': 11359, 'Types': 2634, 'TTR': 0.2319, 'MTLD': 120.14, 'HD-D': 0.8852}

NON-TWAIN CHILD FICTION — NARRATIVE ONLY
{'Tokens': 561064, 'Types': 18841, 'TTR': 0.0336, 'MTLD': 106.66

In [31]:
from pathlib import Path
import pandas as pd

BASE_PATH = Path(
    "/Users/manalaltammami/Desktop/twain_child_corpus"
)

adult_files = [
    "Silas_Lapham_narrative.txt",
    "Ministers_Charge_narrative.txt",
    "Hazard_New_Fortunes_narrative.txt",
    "Portrait_of_a_Lady_narrative.txt",
    "Bostonians_narrative.txt",
    "Washington_Square_narrative.txt",
    "Pointed_Firs_narrative.txt",
    "Deephaven_narrative.txt",
    "Country_Doctor_narrative.txt",
    "Awakening_narrative.txt",
    "Graysons_narrative.txt",
    "McTeague_narrative.txt",
    "Octopus_narrative.txt",
    "Red_Badge_of_Courage_narrative.txt",
    "Maggie_narrative.txt",
    "Main_Travelled_Roads_narrative.txt",
    "Old_Creole_Days_narrative.txt",
    "Conjure_Woman_narrative.txt",
    "Looking_Backward_narrative.txt",
    "Theron_Ware_narrative.txt",
    "Sister_Carrie_narrative.txt",
    "Summer_narrative.txt",
    "Scarlet_Letter_narrative.txt",
    "Seven_Gables_narrative.txt",
    "Moby_Dick_narrative.txt",
    "Aspern_Papers_narrative.txt",
    "Bartleby_narrative.txt",
    "Benito_Cereno_narrative.txt"
]

# Check all files before calculating.
missing = [
    filename for filename in adult_files
    if not (BASE_PATH / filename).is_file()
]

if missing:
    raise FileNotFoundError(
        "Missing narrative files:\n" + "\n".join(missing)
    )

assert len(adult_files) == 28
assert len(set(adult_files)) == 28

adult_tokens = {}
adult_results = {}
combined_adult_tokens = []

for filename in adult_files:
    with (BASE_PATH / filename).open(encoding="utf-8") as f:
        text = f.read()

    tokens = tokenize(text)

    adult_tokens[filename] = tokens
    adult_results[filename] = lexical_metrics(tokens)
    combined_adult_tokens.extend(tokens)

df_adult = pd.DataFrame.from_dict(
    adult_results,
    orient="index"
)
df_adult.index.name = "Text"

print(df_adult.to_string())

adult_combined_results = lexical_metrics(combined_adult_tokens)

print("\nADULT CORPUS — NARRATIVE ONLY")
print(adult_combined_results)

df_adult.to_csv(
    BASE_PATH / "adult_narrative_lexical_results.csv"
)

                                    Tokens  Types     TTR    MTLD    HD-D
Text                                                                     
Silas_Lapham_narrative.txt          129245   8382  0.0649   87.63  0.8686
Ministers_Charge_narrative.txt      115995   8424  0.0726   88.29  0.8662
Hazard_New_Fortunes_narrative.txt   163596  11096  0.0678   88.46  0.8688
Portrait_of_a_Lady_narrative.txt    225399  10724  0.0476   88.40  0.8699
Bostonians_narrative.txt            162770  10120  0.0622   88.39  0.8597
Washington_Square_narrative.txt      64180   5492  0.0856   81.12  0.8616
Pointed_Firs_narrative.txt           41980   5018  0.1195  117.77  0.8723
Deephaven_narrative.txt              54924   5607  0.1021   94.42  0.8594
Country_Doctor_narrative.txt         89552   7228  0.0807  103.74  0.8609
Awakening_narrative.txt              49937   5883  0.1178   90.51  0.8637
Graysons_narrative.txt               91482   9308  0.1017  101.05  0.8691
McTeague_narrative.txt              11